In [55]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [56]:
df = read_table("""
    SELECT state, year, unemp_rate, unemp_graduate,qualification
    FROM sc_bronze.dosm_graduates_state
    """)

In [57]:
df.head(20)

,state,year,unemp_rate,unemp_graduate,qualification
0,Johor,2016,5.4,8200.0,degree
1,Johor,2016,4.7,7500.0,diploma
2,Johor,2017,4.2,7200.0,degree
3,Johor,2017,4.6,8300.0,diploma
4,Johor,2018,4.1,8200.0,degree
5,Johor,2018,2.5,5400.0,diploma
6,Johor,2019,3.9,7900.0,degree
7,Johor,2019,3.2,6700.0,diploma
8,Johor,2020,5.4,10400.0,degree
9,Johor,2020,3.6,7600.0,diploma


In [58]:
"""
Graduate Unemployment Forecasting — v3 (AutoML Tuning Add-on)
=============================================================
NEW in v3:
  - AutoMLTuner class: unified Optuna tuning across all tree models
  - Ensemble blending: top-N models weighted by inverse-RMSE
  - Dynamic trial budget: more trials to historically stronger models
  - Per-group time cap: max_seconds_per_group prevents runaway tuning
  - Tuning summary report: which models won, avg improvement over baseline
"""

import warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from tqdm import tqdm
import plotly.graph_objects as go
from collections import defaultdict

# ============================================================
# AutoML CONFIG  ← tweak these to control tuning behaviour
# ============================================================
AUTOML_CFG = dict(
    total_trials        = 60,    # total Optuna trials split across all tree models
    max_seconds_per_group = 30,  # hard wall-clock cap per group (seconds)
    top_n_ensemble      = 3,     # blend top-N models by inverse-RMSE weight
    min_improvement_pct = 1.0,   # only accept tuned model if RMSE improves by ≥ 1 %
    retrain_on_all_data = True,  # retrain winning model on full data before forecasting
)

# ============================================================
# AutoMLTuner  — core add-on class
# ============================================================
class AutoMLTuner:
    """
    Runs Optuna HPO across CatBoost / XGB / LGBM / GradBoost simultaneously.
    Allocates the trial budget dynamically: models that historically performed
    better in earlier groups get proportionally more trials.

    Parameters
    ----------
    cfg : dict
        AutoML config (see AUTOML_CFG above).
    model_win_history : defaultdict(int)
        Shared counter updated after each group so trial allocation improves
        over the run (warm-start exploration).
    """

    MODEL_NAMES = ["CatBoost", "XGB", "LGBM", "GradBoost"]

    def __init__(self, cfg: dict, model_win_history: defaultdict):
        self.cfg     = cfg
        self.history = model_win_history

    # ---- per-model search spaces ----------------------------------------
    @staticmethod
    def _catboost_space(trial):
        return dict(
            iterations    = trial.suggest_int("iterations", 100, 600),
            depth         = trial.suggest_int("depth", 3, 8),
            learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            l2_leaf_reg   = trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
            bagging_temperature = trial.suggest_float("bagging_temperature", 0.0, 1.0),
            random_strength     = trial.suggest_float("random_strength", 0.0, 2.0),
        )

    @staticmethod
    def _xgb_space(trial):
        return dict(
            n_estimators     = trial.suggest_int("n_estimators", 50, 400),
            max_depth        = trial.suggest_int("max_depth", 2, 8),
            learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            subsample        = trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_alpha        = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            reg_lambda       = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            min_child_weight = trial.suggest_int("min_child_weight", 1, 10),
        )

    @staticmethod
    def _lgbm_space(trial):
        return dict(
            n_estimators  = trial.suggest_int("n_estimators", 50, 400),
            max_depth     = trial.suggest_int("max_depth", 2, 8),
            learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            num_leaves    = trial.suggest_int("num_leaves", 15, 127),
            subsample     = trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_alpha     = trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            reg_lambda    = trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            min_child_samples = trial.suggest_int("min_child_samples", 1, 20),
        )

    @staticmethod
    def _gradboost_space(trial):
        return dict(
            n_estimators  = trial.suggest_int("n_estimators", 50, 300),
            max_depth     = trial.suggest_int("max_depth", 2, 6),
            learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            subsample     = trial.suggest_float("subsample", 0.5, 1.0),
            max_features  = trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
            min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10),
        )

    SPACES = {
        "CatBoost": _catboost_space.__func__,
        "XGB":      _xgb_space.__func__,
        "LGBM":     _lgbm_space.__func__,
        "GradBoost":_gradboost_space.__func__,
    }

    # ---- build model from params ----------------------------------------
    @staticmethod
    def _build(name, params):
        if name == "CatBoost":
            return CatBoostRegressor(**params, verbose=0,
                                      allow_writing_files=False, random_seed=42)
        if name == "XGB":
            return XGBRegressor(**params, tree_method="hist",
                                 verbosity=0, random_state=42)
        if name == "LGBM":
            return LGBMRegressor(**params, n_jobs=-1,
                                  verbose=-1, random_state=42)
        return GradientBoostingRegressor(**params, random_state=42)

    # ---- dynamic trial allocation ----------------------------------------
    def _allocate_trials(self):
        """
        Distribute self.cfg['total_trials'] across models.
        Models with more historical wins get proportionally more trials.
        Minimum 5 trials per model to keep exploration alive.
        """
        total = self.cfg["total_trials"]
        wins  = {m: self.history[m] + 1 for m in self.MODEL_NAMES}  # +1 = Laplace smooth
        total_wins = sum(wins.values())
        alloc = {m: max(5, round(total * wins[m] / total_wins))
                 for m in self.MODEL_NAMES}
        # re-normalise if rounding pushed over budget
        scale = total / sum(alloc.values())
        alloc = {m: max(5, round(v * scale)) for m, v in alloc.items()}
        return alloc

    # ---- main entry point ------------------------------------------------
    def tune(self, X_tr, y_tr, X_val, y_val,
             baseline_scores: dict) -> dict:
        """
        Returns dict: {model_name: (rmse, fitted_model, best_params)}
        Only includes models that improved over baseline by ≥ min_improvement_pct.
        """
        alloc     = self._allocate_trials()
        improved  = {}
        deadline  = time.time() + self.cfg["max_seconds_per_group"]

        for name in self.MODEL_NAMES:
            if time.time() > deadline:
                break  # hard time cap

            n_trials = alloc[name]
            space_fn = self.SPACES[name]

            def objective(trial, _name=name, _space=space_fn):
                if time.time() > deadline:
                    raise optuna.exceptions.TrialPruned()
                params = _space(trial)
                model  = self._build(_name, params)
                model.fit(X_tr, y_tr)
                return np.sqrt(mean_squared_error(y_val, model.predict(X_val)))

            study = optuna.create_study(direction="minimize",
                                         sampler=optuna.samplers.TPESampler(seed=42))
            study.optimize(objective, n_trials=n_trials,
                           show_progress_bar=False, catch=(Exception,))

            if not study.trials:
                continue

            best_params = study.best_params
            best_rmse   = study.best_value
            baseline    = baseline_scores.get(name, float("inf"))
            improvement = (baseline - best_rmse) / (baseline + 1e-9) * 100

            if improvement >= self.cfg["min_improvement_pct"]:
                model = self._build(name, best_params)
                model.fit(X_tr, y_tr)
                improved[name] = (best_rmse, model, best_params)

        return improved

    # ---- ensemble blending -----------------------------------------------
    @staticmethod
    def blend(model_score_dict: dict, X: np.ndarray,
              top_n: int = 3) -> np.ndarray:
        """
        Weighted average of top-N model predictions, weighted by 1/RMSE.
        model_score_dict: {name: (rmse, fitted_model, params)}
        """
        sorted_models = sorted(model_score_dict.items(),
                                key=lambda kv: kv[1][0])[:top_n]
        preds   = []
        weights = []
        for _, (rmse, model, _) in sorted_models:
            preds.append(model.predict(X))
            weights.append(1.0 / (rmse + 1e-9))
        weights = np.array(weights) / sum(weights)
        return sum(w * p for w, p in zip(weights, preds))

    # ---- forecast helper -------------------------------------------------
    @staticmethod
    def forecast_tree(model, last_row: pd.Series,
                      rolling_window: list, n_steps: int = 5) -> list:
        """Recursive 1-step-ahead forecast for tree models."""
        last_row      = last_row.copy()
        rolling_window = list(rolling_window)
        future_vals   = []
        for _ in range(n_steps):
            pred = model.predict(last_row.values.reshape(1, -1))[0]
            future_vals.append(pred)
            rolling_window.append(pred)
            rolling_window = rolling_window[-3:]
            last_row["lag2"]       = last_row["lag1"]
            last_row["lag1"]       = pred
            last_row["trend_diff"] = pred - last_row["lag1"]
            last_row["roll_mean3"] = np.mean(rolling_window)
            last_row["roll_std3"]  = (np.std(rolling_window)
                                       if len(rolling_window) > 1 else 0)
            last_row["year"]      += 1
        return future_vals


# ============================================================
# 1. LOAD & FEATURE ENGINEERING  (unchanged)
# ============================================================
df["year"] = pd.to_numeric(df["year"])
df = df.sort_values(["state", "qualification", "year"])

le_state = LabelEncoder()
le_qual  = LabelEncoder()
df["state_enc"] = le_state.fit_transform(df["state"])
df["qual_enc"]  = le_qual.fit_transform(df["qualification"])

TARGET = "unemp_rate"

g = df.groupby(["state", "qualification"])[TARGET]
df["lag1"]       = g.shift(1)
df["lag2"]       = g.shift(2)
df["roll_mean3"] = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
df["roll_std3"]  = g.transform(lambda x: x.shift(1).rolling(3, min_periods=1).std().fillna(0))
df["trend_diff"] = g.shift(1).diff()
df = df.dropna(subset=["lag1", "lag2"])
df["trend_diff"] = df["trend_diff"].fillna(0)

FEATURES = ["year","state_enc","qual_enc","lag1","lag2","roll_mean3","roll_std3","trend_diff"]

# ============================================================
# 2. STORAGE
# ============================================================
results, all_preds, future_all = [], [], []
model_win_history  = defaultdict(int)   # fed into AutoMLTuner for dynamic allocation
tuning_improvement = []                 # track automl gains

groups = list(df.groupby(["state", "qualification"]))
tuner  = AutoMLTuner(AUTOML_CFG, model_win_history)

# ============================================================
# 3. TRAIN LOOP
# ============================================================
pbar = tqdm(groups, desc="Training", unit="group", dynamic_ncols=True)

for (state, qual), group in pbar:
    pbar.set_description(f"{state} | {qual}")

    if len(group) < 6:
        continue

    group = group.sort_values("year").reset_index(drop=True)
    X     = group[FEATURES]
    y     = group[TARGET]
    split = int(len(group) * 0.8)
    if split < 2:
        continue

    X_train, X_test = X.iloc[:split], X.iloc[split:]
    y_train, y_test = y.iloc[:split], y.iloc[split:]

    # ---- Baseline models (same as v2) ----
    baseline_scores  = {}
    baseline_models  = {}

    for name, ModelCls, kw in [
        ("CatBoost",  CatBoostRegressor,
            dict(iterations=200, depth=5, learning_rate=0.1,
                 verbose=0, allow_writing_files=False, random_seed=42)),
        ("XGB",       XGBRegressor,
            dict(n_estimators=100, max_depth=4, learning_rate=0.1,
                 tree_method="hist", verbosity=0, random_state=42)),
        ("LGBM",      LGBMRegressor,
            dict(n_estimators=100, max_depth=4, learning_rate=0.1,
                 n_jobs=-1, verbose=-1, random_state=42)),
        ("GradBoost", GradientBoostingRegressor,
            dict(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)),
    ]:
        try:
            m = ModelCls(**kw)
            m.fit(X_train, y_train)
            rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test)))
            baseline_scores[name]  = rmse
            baseline_models[name]  = m
        except Exception:
            pass

    # ---- Holt ETS baseline ----
    try:
        trend_type = "add" if len(y_train) >= 4 else None
        ets = ExponentialSmoothing(y_train.values, trend=trend_type,
                                    initialization_method="estimated").fit(optimized=True)
        ets_rmse = np.sqrt(mean_squared_error(y_test, ets.forecast(len(y_test))))
        baseline_scores["HoltETS"] = ets_rmse
        baseline_models["HoltETS"] = ets
    except Exception:
        pass

    if not baseline_scores:
        continue

    # ---- AutoML tuning ----
    tuned_scores = {}
    if split >= 4:
        tuned_scores = tuner.tune(
            X_train.values, y_train.values,
            X_test.values,  y_test.values,
            baseline_scores
        )

    # ---- Merge baseline + tuned; pick ensemble or single best ----
    all_scores = {**{n: (r, baseline_models[n], {})
                     for n, r in baseline_scores.items()},
                  **tuned_scores}

    # Try ensemble blend (tree models only)
    tree_candidates = {k: v for k, v in all_scores.items()
                       if k in AutoMLTuner.MODEL_NAMES}

    final_pred_test = None
    final_name      = None
    final_rmse      = float("inf")

    if len(tree_candidates) >= 2 and AUTOML_CFG["top_n_ensemble"] > 1:
        try:
            blend_pred = AutoMLTuner.blend(
                tree_candidates, X_test.values,
                top_n=min(AUTOML_CFG["top_n_ensemble"], len(tree_candidates))
            )
            blend_rmse = np.sqrt(mean_squared_error(y_test, blend_pred))
            if blend_rmse < min(v[0] for v in tree_candidates.values()):
                final_pred_test = blend_pred
                final_name      = "Ensemble"
                final_rmse      = blend_rmse
        except Exception:
            pass

    if final_name is None:
        best_name = min(all_scores, key=lambda k: all_scores[k][0])
        final_rmse, best_model, _ = all_scores[best_name]
        final_name  = best_name
        if best_name in ("CatBoost","XGB","LGBM","GradBoost"):
            final_pred_test = best_model.predict(X_test)
        else:
            final_pred_test = best_model.forecast(len(y_test))

    # ---- Track improvement ----
    naive_best_rmse = min(baseline_scores.values())
    improvement_pct = (naive_best_rmse - final_rmse) / (naive_best_rmse + 1e-9) * 100
    tuning_improvement.append({
        "state": state, "qualification": qual,
        "baseline_rmse": round(naive_best_rmse, 4),
        "automl_rmse":   round(final_rmse, 4),
        "improvement_pct": round(improvement_pct, 2),
        "winner": final_name,
    })
    model_win_history[final_name] += 1

    pbar.set_postfix(winner=final_name, rmse=f"{final_rmse:.4f}",
                     gain=f"{improvement_pct:+.1f}%")

    results.append({"state": state, "qualification": qual,
                     "best_model": final_name, "rmse": final_rmse})

    # ---- Store test predictions ----
    temp = group.copy()
    temp["pred"] = np.nan
    temp.iloc[split:, temp.columns.get_loc("pred")] = final_pred_test
    temp["model"] = final_name
    all_preds.append(temp)

    # ---- 5-year forecast ----
    last_year = group["year"].max()

    if final_name == "Ensemble":
        # Blend ensemble: average recursive forecasts from each tree model
        top_models = sorted(tree_candidates.items(),
                             key=lambda kv: kv[1][0])[:AUTOML_CFG["top_n_ensemble"]]
        weights = np.array([1 / (v[0] + 1e-9) for _, v in top_models])
        weights /= weights.sum()

        # Re-fit each on full data
        ensemble_futures = []
        for (m_name, (_, m_model, m_params)), w in zip(top_models, weights):
            fm = AutoMLTuner._build(m_name, m_params) if m_params else m_model
            fm.fit(X, y)
            fvals = AutoMLTuner.forecast_tree(
                fm, X.iloc[-1].copy(), list(y.values[-3:]), n_steps=5
            )
            ensemble_futures.append(np.array(fvals) * w)
        future_vals = list(sum(ensemble_futures))

    elif final_name in AutoMLTuner.MODEL_NAMES:
        _, best_model_obj, best_params = all_scores[final_name]
        fm = AutoMLTuner._build(final_name, best_params) if best_params else best_model_obj
        if AUTOML_CFG["retrain_on_all_data"]:
            fm.fit(X, y)
        future_vals = AutoMLTuner.forecast_tree(
            fm, X.iloc[-1].copy(), list(y.values[-3:]), n_steps=5
        )

    else:  # HoltETS
        trend_type = "add" if len(y) >= 4 else None
        fm = ExponentialSmoothing(y.values, trend=trend_type,
                                   initialization_method="estimated").fit(optimized=True)
        future_vals = list(fm.forecast(5))

    for i, val in enumerate(future_vals):
        future_all.append({"state": state, "qualification": qual,
                            "year": last_year + i + 1, "forecast": val})

# ============================================================
# 4. SUMMARY
# ============================================================
results_df    = pd.DataFrame(results)
pred_df       = pd.concat(all_preds, ignore_index=True)
future_df     = pd.DataFrame(future_all)
improvement_df = pd.DataFrame(tuning_improvement)

print("\n===== MODEL WIN COUNT =====")
print(results_df["best_model"].value_counts().to_string())

print("\n===== RMSE STATS =====")
print(results_df.groupby("best_model")["rmse"]
      .agg(["mean","median","min","max"]).round(4).to_string())

print(f"\nOverall — Mean RMSE : {results_df['rmse'].mean():.4f} "
      f"| Median : {results_df['rmse'].median():.4f}")

print("\n===== AutoML IMPROVEMENT SUMMARY =====")
print(f"Groups improved by AutoML : "
      f"{(improvement_df['improvement_pct'] > 0).sum()} / {len(improvement_df)}")
print(f"Avg improvement           : "
      f"{improvement_df['improvement_pct'].mean():.2f}%")
print(f"Best single-group gain    : "
      f"{improvement_df['improvement_pct'].max():.2f}%")
print("\nTop 10 most-improved groups:")
print(improvement_df.nlargest(10, "improvement_pct")
      [["state","qualification","baseline_rmse","automl_rmse","improvement_pct","winner"]]
      .to_string(index=False))

# ============================================================
# 5. VISUALIZATION  (unchanged from v2, model label updated)
# ============================================================
for (state, qual), group in pred_df.groupby(["state", "qualification"]):
    future_group = future_df[
        (future_df["state"] == state) & (future_df["qualification"] == qual)
    ]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=group["year"], y=group[TARGET],
        mode="lines+markers", name="Actual", line=dict(color="#1f77b4")))
    fig.add_trace(go.Scatter(x=group["year"], y=group["pred"],
        mode="lines+markers", name="Predicted", line=dict(color="#ff7f0e")))
    fig.add_trace(go.Scatter(x=future_group["year"], y=future_group["forecast"],
        mode="lines+markers", name="Forecast (5 yr)",
        line=dict(color="#2ca02c", dash="dash")))
    fig.update_layout(
        title=f"{state} — {qual} | {group['model'].iloc[0]}",
        xaxis_title="Year", yaxis_title="Graduate Unemployment Rate (%)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white"
    )
    fig.show()

# ============================================================
# 6. EXPORT (Historical + Forecast Combined)
# ============================================================

# ---- 1. Prepare historical (actual) data ----
hist_df = df[["state", "qualification", "year", TARGET]].copy()
hist_df = hist_df.rename(columns={TARGET: "value"})
hist_df["type"] = "actual"

# ---- 2. Prepare forecast data ----
fc_df = future_df.copy()
fc_df = fc_df.rename(columns={"forecast": "value"})
fc_df["type"] = "forecast"

# ---- 3. Combine ----
combined_df = pd.concat([hist_df, fc_df], ignore_index=True)

# Standardize qualification naming
combined_df["qualification"] = combined_df["qualification"].str.lower()

# ---- 4. Pivot (degree & diploma side-by-side) ----
pivot_df = combined_df.pivot_table(
    index=["state", "year", "type"],
    columns="qualification",
    values="value"
).reset_index()

# Ensure both exist
if "degree" not in pivot_df.columns:
    pivot_df["degree"] = np.nan
if "diploma" not in pivot_df.columns:
    pivot_df["diploma"] = np.nan

# ---- 5. Compute unemployment rate ----
pivot_df["unemployment_rate"] = pivot_df[["degree", "diploma"]].mean(axis=1)

# ---- 6. Final formatting ----
final_df = pivot_df.rename(columns={
    "state": "State",
    "year": "Year",
    "type": "Data_Type"
})[["State", "Year", "Data_Type", "unemployment_rate"]]

# Sort
final_df = final_df.sort_values(["State", "Year", "Data_Type"]).reset_index(drop=True)

# ---- 7. Save ----
final_df.to_csv("graduate_unemployment_full.csv", index=False)

print("\n✅ Exported: graduate_unemployment_full.csv")
print(final_df.head(15))

W.P. Putrajaya | diploma: 100%|██████████| 32/32 [00:55<00:00,  1.73s/group, gain=+7.0%, rmse=0.3869, winner=GradBoost]    


===== MODEL WIN COUNT =====
best_model
GradBoost    17
HoltETS      12
XGB           3

===== RMSE STATS =====
              mean  median     min     max
best_model                                
GradBoost   0.4956  0.3869  0.0273  1.3325
HoltETS     0.2679  0.3590  0.0000  0.4423
XGB         0.7608  0.7122  0.6079  0.9625

Overall — Mean RMSE : 0.4351 | Median : 0.3801

===== AutoML IMPROVEMENT SUMMARY =====
Groups improved by AutoML : 16 / 32
Avg improvement           : 10.35%
Best single-group gain    : 72.70%

Top 10 most-improved groups:
            state qualification  baseline_rmse  automl_rmse  improvement_pct    winner
           Melaka       diploma         0.1000       0.0273            72.70 GradBoost
         Selangor       diploma         0.0860       0.0502            41.62 GradBoost
           Pahang       diploma         0.3883       0.2302            40.71 GradBoost
           Perlis        degree         0.5758       0.3511            39.03 GradBoost
     Pulau Pin


✅ Exported: graduate_unemployment_full.csv
qualification  State  Year Data_Type  unemployment_rate
0              Johor  2018    actual           3.300000
1              Johor  2019    actual           3.550000
2              Johor  2020    actual           4.500000
3              Johor  2021    actual           4.100000
4              Johor  2022    actual           3.900000
5              Johor  2023    actual           3.250000
6              Johor  2024    actual           3.250000
7              Johor  2025  forecast           3.515810
8              Johor  2026  forecast           4.108479
9              Johor  2027  forecast           4.093068
10             Johor  2028  forecast           3.765803
11             Johor  2029  forecast           3.415813
12             Kedah  2018    actual           3.300000
13             Kedah  2019    actual           4.150000
14             Kedah  2020    actual           4.100000


In [59]:
# Write to sc_silver.underemployment_forecast
try:
    write_table(final_df, 'sc_silver', 'unemployment_forecast')
    print("Successfully written to Supabase: sc_silver.unemployment_forecast")
except Exception as e:
    print(f"Failed to write to Supabase: {e}")

Table sc_silver.unemployment_forecast written successfully.
Successfully written to Supabase: sc_silver.unemployment_forecast
